# 01 Preprocessing
## 전처리

AMEX customer-month records를 customer-level modeling table로 바꾸는 단계다.


## Setup
### 설정


In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def read_table(name):
    return pd.read_csv(TABLE_DIR / name)

def save_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path

def pct(series, digits=2):
    return (series.astype(float) * 100).round(digits)


## Source table
### 원자료 단위


In [2]:
source_table = pd.DataFrame(
    [
        {"table": "train.parquet", "unit": "customer-month", "rows": 5_531_451, "columns": 190},
        {"table": "test.parquet", "unit": "customer-month", "rows": 11_363_762, "columns": 190},
        {"table": "train_labels.csv", "unit": "customer", "rows": 458_913, "columns": 2},
    ]
)
display(source_table)


,table,unit,rows,columns
0,train.parquet,customer-month,5531451,190
1,test.parquet,customer-month,11363762,190
2,train_labels.csv,customer,458913,2


## Feature layers
### 변수 생성 구조


In [3]:
feature_layers = pd.DataFrame(
    [
        {
            "layer": "Base",
            "features": "last, first, mean, std, min, max, sum, median, count",
            "signal": "long-term customer level and dispersion",
        },
        {
            "layer": "Change",
            "features": "last - mean, last - first, last / mean, last / first",
            "signal": "recent deviation from customer history",
        },
        {
            "layer": "Temporal",
            "features": "last - lag1, last - lag3, last - lag6, recent_3m_mean, recent_6m_mean, recent_3m_std",
            "signal": "short-term movement and momentum",
        },
        {
            "layer": "Missing",
            "features": "missing count, missing ratio, variable-level missing flags",
            "signal": "information availability and reporting pattern",
        },
        {
            "layer": "Categorical",
            "features": "last, first, nunique, mode",
            "signal": "latest status and customer state changes",
        },
    ]
)
display(feature_layers)


,layer,features,signal
0,Base,"last, first, mean, std, min, max, sum, median,...",long-term customer level and dispersion
1,Change,"last - mean, last - first, last / mean, last /...",recent deviation from customer history
2,Temporal,"last - lag1, last - lag3, last - lag6, recent_...",short-term movement and momentum
3,Missing,"missing count, missing ratio, variable-level m...",information availability and reporting pattern
4,Categorical,"last, first, nunique, mode",latest status and customer state changes


## Customer-level transformation
### 고객 단위 변환


In [4]:
customer_month_demo = pd.DataFrame(
    {
        "customer_ID": ["C001"] * 4 + ["C002"] * 4 + ["C003"] * 4,
        "statement_order": [1, 2, 3, 4] * 3,
        "B_1": [0.20, 0.21, 0.18, 0.35, 0.05, 0.06, 0.07, 0.06, 0.44, 0.50, 0.63, 0.72],
        "D_39": [0, 1, 0, 2, 0, 0, 0, 1, 3, 4, 4, 5],
        "S_3": [0.12, 0.11, 0.14, 0.18, 0.03, np.nan, 0.04, 0.05, 0.21, 0.25, 0.31, 0.33],
        "target": [0] * 4 + [0] * 4 + [1] * 4,
    }
)

numeric_cols = ["B_1", "D_39", "S_3"]
grouped = customer_month_demo.sort_values(["customer_ID", "statement_order"]).groupby("customer_ID")

base = grouped[numeric_cols].agg(["last", "first", "mean", "std", "min", "max"])
base.columns = [f"{col}_{stat}" for col, stat in base.columns]

change = pd.DataFrame(index=base.index)
temporal = pd.DataFrame(index=base.index)
for col in numeric_cols:
    change[f"{col}_last_minus_mean"] = base[f"{col}_last"] - base[f"{col}_mean"]
    change[f"{col}_last_minus_first"] = base[f"{col}_last"] - base[f"{col}_first"]
    change[f"{col}_last_div_mean"] = base[f"{col}_last"] / base[f"{col}_mean"].replace(0, np.nan)

    last = grouped[col].last()
    lag1 = grouped[col].nth(-2)
    recent_3m_mean = grouped[col].tail(3).groupby(customer_month_demo["customer_ID"]).mean()
    temporal[f"{col}_last_minus_lag1"] = last - lag1
    temporal[f"{col}_recent_3m_mean"] = recent_3m_mean

missing = grouped[numeric_cols].apply(lambda x: x.isna().mean()).add_suffix("_missing_rate")
target = grouped["target"].last()

model_table_demo = pd.concat([base, change, temporal, missing, target.rename("target")], axis=1).reset_index()
display(model_table_demo)
print("model_table_shape:", model_table_demo.shape)


,customer_ID,B_1_last,B_1_first,B_1_mean,B_1_std,B_1_min,B_1_max,D_39_last,D_39_first,D_39_mean,D_39_std,D_39_min,D_39_max,S_3_last,S_3_first,S_3_mean,S_3_std,S_3_min,S_3_max,B_1_last_minus_mean,B_1_last_minus_first,B_1_last_div_mean,D_39_last_minus_mean,D_39_last_minus_first,D_39_last_div_mean,S_3_last_minus_mean,S_3_last_minus_first,S_3_last_div_mean,B_1_last_minus_lag1,B_1_recent_3m_mean,D_39_last_minus_lag1,D_39_recent_3m_mean,S_3_last_minus_lag1,S_3_recent_3m_mean,B_1_missing_rate,D_39_missing_rate,S_3_missing_rate,target
0,C001,0.35,0.20,0.2350,0.077675,0.18,0.35,2,0,0.75,0.957427,0,2,0.18,0.12,0.1375,0.030957,0.11,0.18,0.1150,0.15,1.489362,1.25,2,2.666667,0.0425,0.06,1.309091,NaN,0.246667,NaN,1.000000,NaN,0.143333,0.0,0.0,0.00,0
1,C002,0.06,0.05,0.0600,0.008165,0.05,0.07,1,0,0.25,0.500000,0,1,0.05,0.03,0.0400,0.010000,0.03,0.05,0.0000,0.01,1.000000,0.75,1,4.000000,0.0100,0.02,1.250000,NaN,0.063333,NaN,0.333333,NaN,0.045000,0.0,0.0,0.25,0
2,C003,0.72,0.44,0.5725,0.126326,0.44,0.72,5,3,4.00,0.816497,3,5,0.33,0.21,0.2750,0.055076,0.21,0.33,0.1475,0.28,1.257642,1.00,2,1.250000,0.0550,0.12,1.200000,NaN,0.616667,NaN,4.333333,NaN,0.296667,0.0,0.0,0.00,1


model_table_shape: (3, 38)


## Model matrix
### 모델 입력 행렬


In [5]:
X_demo = model_table_demo.drop(columns=["customer_ID", "target"])
y_demo = model_table_demo["target"].astype(int)

model_matrix_summary = pd.DataFrame(
    {
        "item": ["customers", "features", "default_rate", "missing_cells"],
        "value": [len(model_table_demo), X_demo.shape[1], round(y_demo.mean(), 4), int(X_demo.isna().sum().sum())],
    }
)
display(model_matrix_summary)
display(X_demo.head())


,item,value
0,customers,3.0000
1,features,36.0000
2,default_rate,0.3333
3,missing_cells,9.0000


,B_1_last,B_1_first,B_1_mean,B_1_std,B_1_min,B_1_max,D_39_last,D_39_first,D_39_mean,D_39_std,D_39_min,D_39_max,S_3_last,S_3_first,S_3_mean,S_3_std,S_3_min,S_3_max,B_1_last_minus_mean,B_1_last_minus_first,B_1_last_div_mean,D_39_last_minus_mean,D_39_last_minus_first,D_39_last_div_mean,S_3_last_minus_mean,S_3_last_minus_first,S_3_last_div_mean,B_1_last_minus_lag1,B_1_recent_3m_mean,D_39_last_minus_lag1,D_39_recent_3m_mean,S_3_last_minus_lag1,S_3_recent_3m_mean,B_1_missing_rate,D_39_missing_rate,S_3_missing_rate
0,0.35,0.20,0.2350,0.077675,0.18,0.35,2,0,0.75,0.957427,0,2,0.18,0.12,0.1375,0.030957,0.11,0.18,0.1150,0.15,1.489362,1.25,2,2.666667,0.0425,0.06,1.309091,NaN,0.246667,NaN,1.000000,NaN,0.143333,0.0,0.0,0.00
1,0.06,0.05,0.0600,0.008165,0.05,0.07,1,0,0.25,0.500000,0,1,0.05,0.03,0.0400,0.010000,0.03,0.05,0.0000,0.01,1.000000,0.75,1,4.000000,0.0100,0.02,1.250000,NaN,0.063333,NaN,0.333333,NaN,0.045000,0.0,0.0,0.25
2,0.72,0.44,0.5725,0.126326,0.44,0.72,5,3,4.00,0.816497,3,5,0.33,0.21,0.2750,0.055076,0.21,0.33,0.1475,0.28,1.257642,1.00,2,1.250000,0.0550,0.12,1.200000,NaN,0.616667,NaN,4.333333,NaN,0.296667,0.0,0.0,0.00


전처리의 결과물은 customer-level feature matrix다. 이후 모델은 이 행렬에서 default risk score를 만든다.
